# NB_lab_test_trends
## Silver → Gold |  Health Network

**Target Table:** `adf_healthgroup_metadatadriven.gold.lab_test_trends`  
**Grain:** One row per test type (`test_name`) per calendar month  
**Load Type:** Full rebuild — `CREATE OR REPLACE TABLE … AS SELECT`  
**Source:** STTM `sttm_silver_to_gold.xlsx`, Sheet `lab_test_trends`

### Source Tables
| Table | Role |
|---|---|
| `adf_healthgroup_metadatadriven.silver.lab_results` | Driving table; grouped by `test_name`, `year(test_date)`, `month(test_date)` |
| `adf_healthgroup_metadatadriven.silver.calendar` | Optional LEFT JOIN for `month_name` and `quarter_name`; keyed on first day of month |

### Notes
- Calendar join key: `calendar.date_key` (INT `yyyyMMdd`) joined against the first day of each year-month: `CAST(date_format(date_trunc('month', test_date), 'yyyyMMdd') AS INT)`.
- `quarter_name` is derived from `calendar.quarter` as `'Q' || quarter` since the silver calendar stores quarter as `INT`.
- `abnormal_rate_pct` and `critical_rate_pct` guarded with `NULLIF(total_tests, 0)`.
- `prev_month_abnormal_rate_pct` and `abnormal_rate_change_pct` are `NULL` for a test type’s first observed month.
- Window function: `LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month)`.

In [0]:
# Set the Unity Catalog catalog name used throughout this notebook
catalog = "adf_healthgroup_metadatadriven"
print(f"Using catalog: {catalog}")

In [0]:
%sql
-- ============================================================
-- NB_lab_test_trends
-- Target: adf_healthgroup_metadatadriven.gold.lab_test_trends
-- Grain : One row per test type per calendar month
-- Load  : Full rebuild (CREATE OR REPLACE)
-- ============================================================

CREATE OR REPLACE TABLE adf_healthgroup_metadatadriven.gold.lab_test_trends AS

WITH monthly_base AS (
    -- Aggregate lab results to test_name + year + month grain
    SELECT
        lr.test_name,
        YEAR(lr.test_date)                                                    AS year,
        MONTH(lr.test_date)                                                   AS month,
        -- Calendar join key: first day of the month as INT yyyyMMdd
        CAST(
            date_format(date_trunc('month', lr.test_date), 'yyyyMMdd')
        AS INT)                                                               AS month_start_key,
        COUNT(*)                                                              AS total_tests,
        COUNT(CASE WHEN lr.result_status = 'Abnormal' THEN 1 END)            AS abnormal_test_count,
        COUNT(CASE WHEN lr.result_status = 'Critical' THEN 1 END)            AS critical_test_count
    FROM adf_healthgroup_metadatadriven.silver.lab_results AS lr
    GROUP BY
        lr.test_name,
        YEAR(lr.test_date),
        MONTH(lr.test_date),
        date_trunc('month', lr.test_date)
),

monthly_with_rates AS (
    -- Compute rates and LAG for month-over-month trend
    SELECT
        mb.*,
        ROUND(
            mb.abnormal_test_count / NULLIF(mb.total_tests, 0) * 100.0, 2
        )                                                                     AS abnormal_rate_pct,
        ROUND(
            mb.critical_test_count / NULLIF(mb.total_tests, 0) * 100.0, 2
        )                                                                     AS critical_rate_pct,
        LAG(
            ROUND(mb.abnormal_test_count / NULLIF(mb.total_tests, 0) * 100.0, 2)
        ) OVER (
            PARTITION BY mb.test_name
            ORDER BY mb.year, mb.month
        )                                                                     AS prev_month_abnormal_rate_pct
    FROM monthly_base AS mb
)

SELECT
    mwr.test_name,
    mwr.year,
    mwr.month,

    -- Optional calendar attributes
    c.month_name,
    CASE WHEN c.quarter IS NOT NULL
         THEN 'Q' || CAST(c.quarter AS STRING)
    END                                                                       AS quarter_name,

    -- Metrics
    mwr.total_tests,
    mwr.abnormal_test_count,
    mwr.critical_test_count,
    mwr.abnormal_rate_pct,
    mwr.critical_rate_pct,

    -- Month-over-month trend
    mwr.prev_month_abnormal_rate_pct,
    ROUND(
        mwr.abnormal_rate_pct - mwr.prev_month_abnormal_rate_pct, 2
    )                                                                         AS abnormal_rate_change_pct

FROM monthly_with_rates                                          AS mwr
LEFT JOIN adf_healthgroup_metadatadriven.silver.calendar         AS c
    ON c.date_key = mwr.month_start_key

ORDER BY mwr.test_name, mwr.year, mwr.month

In [0]:
%sql
-- ============================================================
-- Post-load validation
-- ============================================================

SELECT
    COUNT(*)                                                          AS total_rows,
    COUNT(DISTINCT test_name)                                         AS distinct_test_types,
    MIN(year)  AS min_year,
    MAX(year)  AS max_year,
    SUM(CASE WHEN test_name IS NULL THEN 1 END)                       AS null_test_name,
    SUM(CASE WHEN total_tests <= 0  THEN 1 END)                       AS zero_or_negative_tests,
    SUM(CASE WHEN abnormal_rate_pct > 100 THEN 1 END)                 AS rate_over_100_pct,
    SUM(CASE WHEN month_name IS NULL THEN 1 END)                      AS missing_calendar_join
FROM adf_healthgroup_metadatadriven.gold.lab_test_trends

In [0]:
updated_df = spark.table("adf_healthgroup_metadatadriven.gold.lab_test_trends")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))